# 🚚 PROJETO FINAL PÓS-GRADUAÇÃO IA/DADOS: PREDIÇÃO DE RISCO EM SEGURO DE FROTAS PJ
### Simulador do Motor de Decisão de Subscrição no Padrão Trillia RiskPack (Neurotech / B3 / CESAR)

---

### 📌 Objetivos da Solução:
1. **Entendimento de Risco**: Identificar padrões estatísticos e de machine learning de propensão a sinistro em frotas PJ.
2. **Modelo de Scoring**: Pontuar cada empresa e ordená-la em 10 Decis de Risco (`CALC_DECIL_SCORE` 1 a 10).
3. **Motor de Decisão 3-Way**: Aplicar regras de subscrição em formato XML (`politica_credito_frotas.xml`) para rotear em **Aprovado**, **Aceito com Condição (Agravamento + Rastreador GPS)** ou **Recusado Sumário**.
4. **Impacto de Negócio**: Exportar a base final de resultados (`resultado_motor_decisao_frotas.csv`) demonstrando ganho comercial e proteção de margem.

## ⚙️ TAREFA 1: Configuração do Ambiente e Carregamento do Arquivo CSV

Nesta etapa, preparamos as dependências em Python e realizamos a leitura da base de dados do Bureau (`cesar_residencia_trillia_frota_pj_mercado.csv`).
*Se o arquivo não estiver presente no ambiente local do Colab, o script abrirá a janela de upload automaticamente.*

In [6]:
# ==============================================================================
# ⚙️ TAREFA 1: IMPORTS E CARREGAMENTO DA BASE CSV DO BUREAU
# ==============================================================================
import os
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Nome do arquivo de dados esperado do Bureau
csv_filename = "cesar_residencia_trillia_frota_pj_mercado.csv"

# 2. Verificar se o arquivo já está no ambiente do Colab, senão solicitar Upload
if not os.path.exists(csv_filename):
    print(f"⚠️ Arquivo '{csv_filename}' não encontrado no diretório do Colab.")
    print("📥 Por favor, faça o upload do arquivo usando o botão abaixo:")
    from google.colab import files
    uploaded = files.upload()

# 3. Leitura da base de dados do Bureau (Formato TSV separado por Tab \t)
print(f"\n📂 Lendo arquivo de dados: {csv_filename}...")
df_bureau = pd.read_csv(csv_filename, sep='\t')

# 4. Exibir dimensões e estrutura inicial da base
print(f"✅ Base de Dados Carregada com Sucesso!")
print(f"• Total de Empresas/Apólices: {df_bureau.shape[0]:,} linhas")
print(f"• Total de Variáveis do Bureau: {df_bureau.shape[1]} colunas")
print(f"• Taxa de Sinistro Real (ALVO_CASCO): {df_bureau['ALVO_CASCO'].mean():.2%}")
df_bureau.head(3)


⚠️ Arquivo 'cesar_residencia_trillia_frota_pj_mercado (1).csv' não encontrado no diretório do Colab.
📥 Por favor, faça o upload do arquivo usando o botão abaixo:


Saving cesar_residencia_trillia_frota_pj_mercado (1).csv to cesar_residencia_trillia_frota_pj_mercado (1).csv

📂 Lendo arquivo de dados: cesar_residencia_trillia_frota_pj_mercado (1).csv...
✅ Base de Dados Carregada com Sucesso!
• Total de Empresas/Apólices: 3,932 linhas
• Total de Variáveis do Bureau: 102 colunas
• Taxa de Sinistro Real (ALVO_CASCO): 9.69%


,SECAO_CNAE,CATEGORIA_NATUREZA_JURIDICA,PORTE,QTD_FILIAIS_ATIVAS,COMPETICAO_MERCADO_ESTADUAL,QTD_ABERTURAS_NACIONAL_5_ANOS,QTD_ABERTURAS_ESTADUAL_5_ANOS,QTD_ABERTURAS_GEORREFERENCIADO_5_ANOS,QTD_FECHAMENTOS_NACIONAL_5_ANOS,QTD_FECHAMENTOS_ESTADUAL_5_ANOS,...,DISTANCIA_SHOPPING,DISTANCIA_SUPERMERCADO,DISTANCIA_TREM,DISTANCIA_ESTACIONAMENTOS,DISTANCIA_PONTO_TAXI,DISTANCIA_UNIVERSIDADE,DISTANCIA_AREA_RISCO,CNPJ,ANO_MES,ALVO_CASCO
0,G,LTDA,PEQUENA,5.0,5661.0,19949.0,1404.0,24.0,64085.0,4882.0,...,LONGE,MUITO LONGE,LONGE,LONGE,MUITO LONGE,MUITO LONGE,MUITO LONGE,154618823419,202308,1.0
1,G,LTDA,MICRO,1.0,559.0,5063.0,275.0,97.0,7412.0,307.0,...,MEDIO,MEDIO,MUITO LONGE,MEDIO,LONGE,PROXIMO,MEDIO,17179873443,202307,1.0
2,G,LTDA,MEDIA,4.0,2127.0,118617.0,1336.0,667.0,175177.0,1887.0,...,LONGE,MEDIO,PROXIMO,MEDIO,MEDIO,LONGE,LONGE,77309412440,202105,1.0


## 🔍 TAREFA 2: Entendimento dos Dados e Mapeamento de Variáveis Críticas

Nesta etapa, realizamos o diagnóstico completo da base do **Bureau Trillia/Neurotech** (enriquecida com dados PJ e anonimizada):
- **Origem e Dimensões**: 3.932 apólices/empresas e 102 variáveis explicativas.
- **Temporalidade**: 49 meses contínuos (Maio/2020 a Maio/2024).
- **Tipologia das Variáveis**: Categorização em 75+ variáveis geográficas de exposição/distância, além de atributos judiciais, societários e corporativos.
- **Completude e Nulos**: Diagnóstico de ausentes (apenas 3 variáveis com >5% de nulos na base geral).
- **Distribuição da Variável Alvo (`ALVO_CASCO`)**: Avaliação da taxa de sinistro (9,69%), constatação do desbalanceamento de risco (9,3:1) e fundamentação para uso de métricas como AUC-ROC, KS e Lift por Decil.

In [12]:
# ==============================================================================
# 🔍 TAREFA 2: ANÁLISE EXPLORATÓRIA E DIAGNÓSTICO DAS VARIÁVEIS CHAVE
# ==============================================================================

# 1. Origem e Tamanho da Base
total_linhas = len(df_bureau)
total_colunas = len(df_bureau.columns)

# 2. Período Temporal de Análise (ANO_MES)
min_ano_mes = str(df_bureau['ANO_MES'].min())
max_ano_mes = str(df_bureau['ANO_MES'].max())
qtd_meses = df_bureau['ANO_MES'].nunique()

def format_anomes(anomes_str):
    meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    ano = anomes_str[:4]
    mes = meses[int(anomes_str[4:]) - 1]
    return f"{mes}/{ano}"

periodo_str = f"{format_anomes(min_ano_mes)} a {format_anomes(max_ano_mes)} ({qtd_meses} meses contínuos)"

# 3. Tipologia e Categorização das 102 Variáveis
identificadores = ['CNPJ', 'ANO_MES']
alvo_col = ['ALVO_CASCO']
geo_cols = [c for c in df_bureau.columns if any(k in c for k in ['DISTANCIA_', 'CONCENTRACAO_']) or c in ['CONCENTRACAO_AREA_RISCO', 'DISTANCIA_AREA_RISCO']]
judiciais_cols = [c for c in df_bureau.columns if 'PROCESSOS_JUDICIAIS' in c or 'RECLAMACOES' in c]
corp_cols = [c for c in df_bureau.columns if c not in identificadores + alvo_col + geo_cols + judiciais_cols]

# 4. Diagnóstico Geral de Nulos (Missing Values)
null_series = df_bureau.isnull().sum()
cols_gt_5pct = null_series[(null_series / total_linhas) * 100 > 5]

# 5. Distribuição da Variável Alvo e Razão de Desbalanceamento
alvo_counts = df_bureau['ALVO_CASCO'].value_counts()
alvo_pct = df_bureau['ALVO_CASCO'].value_counts(normalize=True) * 100
razao_desbal = alvo_counts[0] / alvo_counts[1]

# Impressão do Painel Executivo (Conforme )
print("==============================================================================")
print("📊 PAINEL DE ENTENDIMENTO DOS DADOS - SEGURO DE FROTAS PJ")
print("==============================================================================")
print(f"• Origem: Bureau Trillia/Neurotech – base de mercado enriquecida com dados PJ, anonimizada")
print(f"• Tamanho: {total_linhas:,} registros (apólices) | {total_colunas} variáveis (colunas)")
print(f"• Período: {periodo_str}")
print(f"• Tipologia: {len(geo_cols)} Variáveis Geográficas, {len(judiciais_cols)} Judiciais/Reclamações, {len(corp_cols)} Corporativas/Mercadológicas, 2 Identificadores/Tempo, 1 Alvo")
print(f"• Completude: Apenas {len(cols_gt_5pct)} variáveis com >5% de nulos ({list(cols_gt_5pct.index)})")
print("\n--- DISTRIBUIÇÃO DA VARIÁVEL ALVO (ALVO_CASCO) ---")
print(f"• Sem Sinistro (Classe 0) : {alvo_counts[0]:>5} empresas ({alvo_pct[0]:.2f}%)")
print(f"• Com Sinistro (Classe 1) : {alvo_counts[1]:>5} empresas ({alvo_pct[1]:.2f}%)")
print(f"• Razão de Desbalanceamento: {razao_desbal:.1f}:1 — Exige métricas como AUC-ROC, KS e Lift por Decil")

# 6. Resumo das Variáveis Críticas para a Política Trillia
var_criticas = [
    'RISCO_LIDERANCA',
    'CONCENTRACAO_AREA_RISCO',
    'QTD_PROCESSOS_JUDICIAIS_REQUERIDO',
    'PORTE_FATURAMENTO_PRESUMIDO',
    'SECAO_CNAE'
]

print("\n--- RESUMO DE PREENCHIMENTO DAS VARIÁVEIS DA POLÍTICA ---")
summary_dict = []
for col in var_criticas:
    if col in df_bureau.columns:
        nulos = df_bureau[col].isnull().sum()
        pct_nulos = (nulos / len(df_bureau)) * 100
        uniquos = df_bureau[col].nunique()
        summary_dict.append({
            'Variavel_Bureau': col,
            'Valores_Preenchidos': len(df_bureau) - nulos,
            'Nulos': nulos,
            '%_Nulos': round(pct_nulos, 2),
            'Valores_Unicos': uniquos
        })

df_summary = pd.DataFrame(summary_dict)
print(df_summary.to_string(index=False))


📊 PAINEL DE ENTENDIMENTO DOS DADOS - SEGURO DE FROTAS PJ
• Origem: Bureau Trillia/Neurotech – base de mercado enriquecida com dados PJ, anonimizada
• Tamanho: 3,932 registros (apólices) | 102 variáveis (colunas)
• Período: Mai/2020 a Mai/2024 (49 meses contínuos)
• Tipologia: 76 Variáveis Geográficas, 4 Judiciais/Reclamações, 19 Corporativas/Mercadológicas, 2 Identificadores/Tempo, 1 Alvo
• Completude: Apenas 3 variáveis com >5% de nulos (['PORTE_FATURAMENTO_PRESUMIDO', 'CONCENTRACAO_SOCIETARIA', 'RISCO_LIDERANCA'])

--- DISTRIBUIÇÃO DA VARIÁVEL ALVO (ALVO_CASCO) ---
• Sem Sinistro (Classe 0) :  3551 empresas (90.31%)
• Com Sinistro (Classe 1) :   381 empresas (9.69%)
• Razão de Desbalanceamento: 9.3:1 — Exige métricas como AUC-ROC, KS e Lift por Decil

--- RESUMO DE PREENCHIMENTO DAS VARIÁVEIS DA POLÍTICA ---
                  Variavel_Bureau  Valores_Preenchidos  Nulos  %_Nulos  Valores_Unicos
                  RISCO_LIDERANCA                 3713    219     5.57               2
    

## 📈 TAREFA 2.1: Análises Exploratórias Estratégicas - Setores CNAE e Evolução Temporal

Nesta etapa, aprofundamos o entendimento exploratório sobre os fatores de risco e comportamento temporal da carteira:
1. **Taxa de Sinistro por Seção CNAE**: Identificação de setores de altíssimo risco (ex: **S - Serviços** com 18,0% e **Q - Saúde** com 17,1%, quase 2x a média de 9,69%), do setor de maior volume (**G - Comércio**, concentrando 33,5% da carteira com 10,8% de sinistro) e de menor risco (**M - Atividades Profissionais** com 4,7%).
2. **Evolução Temporal da Taxa de Sinistro**: Avaliação dos anos de 2021-2022 (~11,7% - maturação completa), 2023 (9,4% - maturação parcial) e 2024 (2,9% - sinistros não maturados).
3. **Impacto no Modelo & Truncamento Temporal**: Diagnóstico de lag de notificação/regulação de sinistros em 2024 (falsos negativos). Diretriz de truncamento temporal obrigatório na etapa de treinamento do motor.

In [13]:
# ==============================================================================
# 📈 TAREFA 2.1: ANÁLISE SETORIAL (CNAE), EVOLUÇÃO TEMPORAL E MATURAÇÃO
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Análise da Taxa de Sinistro por Seção CNAE
cnae_summary = df_bureau.groupby('SECAO_CNAE')['ALVO_CASCO'].agg(
    Total_Empresas='count',
    Qtd_Sinistros='sum',
    Taxa_Sinistro_Pct=lambda x: (x.mean() * 100),
    Pct_Carteira=lambda x: (len(x) / len(df_bureau) * 100)
).reset_index().sort_values(by='Taxa_Sinistro_Pct', ascending=False)

print("==============================================================================")
print("📊 ANÁLISE EXPLORATÓRIA POR SEÇÃO CNAE")
print("==============================================================================")
print(cnae_summary[['SECAO_CNAE', 'Total_Empresas', 'Pct_Carteira', 'Qtd_Sinistros', 'Taxa_Sinistro_Pct']].head(10).to_string(index=False))

# Destaques Específicos do 
taxa_s = cnae_summary[cnae_summary['SECAO_CNAE'] == 'S']['Taxa_Sinistro_Pct'].values[0] if 'S' in cnae_summary['SECAO_CNAE'].values else 0
taxa_q = cnae_summary[cnae_summary['SECAO_CNAE'] == 'Q']['Taxa_Sinistro_Pct'].values[0] if 'Q' in cnae_summary['SECAO_CNAE'].values else 0
row_g = cnae_summary[cnae_summary['SECAO_CNAE'] == 'G']
taxa_g = row_g['Taxa_Sinistro_Pct'].values[0] if len(row_g) > 0 else 0
pct_g = row_g['Pct_Carteira'].values[0] if len(row_g) > 0 else 0
taxa_m = cnae_summary[cnae_summary['SECAO_CNAE'] == 'M']['Taxa_Sinistro_Pct'].values[0] if 'M' in cnae_summary['SECAO_CNAE'].values else 0

print(f"\n💡 DESTAQUES SETORIAIS:")
print(f"• Setor S (Serviços): {taxa_s:.1f}% | Setor Q (Saúde): {taxa_q:.1f}% — Quase 2x a média de 9,69%")
print(f"• Setor G (Comércio): Concentra {pct_g:.1f}% da carteira com {taxa_g:.1f}% de sinistro")
print(f"• Setor M (Atividades Profissionais): Apenas {taxa_m:.1f}% de sinistro (Menor Risco)")

# 2. Evolução Temporal e Diagnóstico de Maturação (Lag 2024)
df_bureau['ANO'] = df_bureau['ANO_MES'].astype(str).str[:4].astype(int)
df_bureau['PERIODO_GRUPO'] = df_bureau['ANO'].map({2020: '2020', 2021: '2021-2022', 2022: '2021-2022', 2023: '2023', 2024: '2024'})

temp_summary = df_bureau.groupby('PERIODO_GRUPO')['ALVO_CASCO'].agg(
    Total_Apolices='count',
    Sinistros='sum',
    Taxa_Sinistro_Pct=lambda x: (x.mean() * 100)
).reindex(['2021-2022', '2023', '2024']).reset_index()

print("\n==============================================================================")
print("📈 EVOLUÇÃO TEMPORAL E MATURAÇÃO DA TAXA DE SINISTRO")
print("==============================================================================")
print(temp_summary.to_string(index=False))
print("\n⚠️ IMPACTO NO MODELO: Dados de 2024 possuem falsos negativos por lag de regulação.")
print("📌 RECOMENDAÇÃO: Aplicar truncamento temporal (Treinar até Jun/2023, testar até Fev/2024).")

# 3. Geração de Gráficos Visuais (Seaborn / Matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Taxa de Sinistro por Seção CNAE
sns.barplot(data=cnae_summary.head(10), x='SECAO_CNAE', y='Taxa_Sinistro_Pct', ax=axes[0], palette='Blues_r')
axes[0].axhline(9.69, color='red', linestyle='--', label='Média Geral (9.69%)')
axes[0].set_title('Taxa de Sinistro por Seção CNAE (Top 10)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa de Sinistro (%)')
axes[0].set_xlabel('Seção CNAE')
axes[0].legend()

# Gráfico 2: Evolução Temporal da Taxa de Sinistro por Ano
ano_summary = df_bureau.groupby('ANO')['ALVO_CASCO'].mean() * 100
axes[1].plot(ano_summary.index, ano_summary.values, marker='o', linewidth=2.5, color='#1f77b4')
axes[1].axhline(9.69, color='red', linestyle='--', label='Média Geral (9.69%)')
axes[1].set_title('Evolução Temporal da Taxa de Sinistro (2020 - 2024)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Taxa de Sinistro (%)')
axes[1].set_xlabel('Ano')
axes[1].set_xticks(ano_summary.index)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


📊 ANÁLISE EXPLORATÓRIA POR SEÇÃO CNAE
SECAO_CNAE  Total_Empresas  Pct_Carteira  Qtd_Sinistros  Taxa_Sinistro_Pct
         T               1      0.025432            1.0         100.000000
         S             100      2.543235           18.0          18.000000
         Q              41      1.042726            7.0          17.073171
         P              20      0.508647            3.0          15.000000
         R               8      0.203459            1.0          12.500000
         A              74      1.881994            8.0          10.810811
         G            1317     33.494405          142.0          10.782080
         C             750     19.074262           77.0          10.266667
         F             248      6.307223           25.0          10.080645
         J              88      2.238047            8.0           9.090909

💡 DESTAQUES SETORIAIS:
• Setor S (Serviços): 18.0% | Setor Q (Saúde): 17.1% — Quase 2x a média de 9,69%
• Setor G (Comércio): Concentra 

## 🤖 TAREFA 3: Arquitetura da Solução, Truncamento Temporal, Filtro Tríplice e Decis

Nesta etapa, implementamos a arquitetura preditiva alinhada ao ecossistema **Trillia RiskPack**:
- **Estratégia & Ferramentas**: Metodologia CRISP-DM em Python 3.12 (Scikit-Learn, Pandas, NumPy, DecisionRunnerNotation) com Random Forest para scoring e Motor XML 3-Way.
- **1. Truncamento Temporal & Split Out-Of-Time (OOT)**:
  - *Descarte de Dados Imaturos*: Removidos 190 registros de Março a Maio/2024 por falha de regulação (falsos negativos).
  - *Amostra de Treino*: Maio/2020 a Junho/2023 (3.162 apólices maduras com 10,66% de sinistro).
  - *Amostra Teste Out-Of-Time (OOT)*: Julho/2023 a Fevereiro/2024 (580 apólices maduras com 6,90% de sinistro).
- **2. Filtro Tríplice de Seleção de Recursos**:
  1. *Missing Rate*: Eliminação de colunas com >50% de nulos.
  2. *Poder Preditivo (MI Score)*: Avaliação de Informação Mútua na amostragem de treino.
  3. *Estabilidade Temporal (KS Drift)*: Teste Kolmogorov-Smirnov entre safras históricas (2021-2022) e safras recentes (2023) para mitigar Data Drift.

In [13]:
# ==============================================================================
# 🤖 TAREFA 3: TRUNCAMENTO TEMPORAL, FILTRO TRÍPLICE E MODELAGEM EM DECIS
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. TRUNCAMENTO TEMPORAL E SPLIT OUT-OF-TIME (OOT)
# Descarte dos 190 registros imaturos de 2024 (Março a Maio/2024)
df_maduro = df_bureau[df_bureau['ANO_MES'] <= 202402].copy()

# Divisão Temporal Out-Of-Time (OOT)
df_train = df_maduro[df_maduro['ANO_MES'] <= 202306].copy()
df_test_oot = df_maduro[df_maduro['ANO_MES'] > 202306].copy()

print("==============================================================================")
print("⏰ DIAGNÓSTICO DE TRUNCAMENTO TEMPORAL E SPLIT OUT-OF-TIME (OOT)")
print("==============================================================================")
print(f"• Base Total do Bureau: {len(df_bureau):,} apólices")
print(f"• Base Madura (até Fev/2024): {len(df_maduro):,} apólices (descartados {len(df_bureau) - len(df_maduro)} registros imaturos de Mar-Mai/2024)")
print(f"• Amostra de Treino (Mai/2020 a Jun/2023): {len(df_train):,} apólices | Sinistros: {df_train['ALVO_CASCO'].sum():.0f} ({df_train['ALVO_CASCO'].mean():.2%})")
print(f"• Amostra Teste OOT (Jul/2023 a Fev/2024): {len(df_test_oot):,} apólices | Sinistros: {df_test_oot['ALVO_CASCO'].sum():.0f} ({df_test_oot['ALVO_CASCO'].mean():.2%})")

# 2. EXECUÇÃO DO FILTRO TRÍPLICE DE SELEÇÃO DE RECURSOS
target_col = 'ALVO_CASCO'
candidate_cols = df_bureau.select_dtypes(include=[np.number]).columns.drop([target_col, 'CNPJ'], errors='ignore')
X_train_cand = df_train[candidate_cols].fillna(0)
y_train = df_train[target_col].fillna(0)

# Filtro 1: Missing Rate (< 50%)
missing_rates = df_train[candidate_cols].isnull().mean()
valid_cols = missing_rates[missing_rates < 0.50].index

# Filtro 2: Informação Mútua (MI Score)
mi_scores = mutual_info_classif(X_train_cand[valid_cols], y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=valid_cols)

# Filtro 3: Teste KS de Estabilidade Temporal (Data Drift 2021-2022 vs 2023)
df_train['ANO'] = df_train['ANO_MES'].astype(str).str[:4].astype(int)
df_hist = df_train[df_train['ANO'].isin([2021, 2022])]
df_recent = df_train[df_train['ANO'] == 2023]
ks_scores = {col: ks_2samp(df_hist[col].dropna(), df_recent[col].dropna())[0] for col in valid_cols}
ks_series = pd.Series(ks_scores)

# Seleção das 14 Principais Variáveis Estáveis
df_filtro = pd.DataFrame({
    'Missing_Rate_%': (missing_rates[valid_cols] * 100).round(2),
    'MI_Score': mi_series[valid_cols].round(4),
    'KS_Drift': ks_series[valid_cols].round(3)
}).sort_values(by='MI_Score', ascending=False)

selected_features = list(df_filtro.head(14).index)

print("\n==============================================================================")
print("🔍 MATRIZ DO FILTRO TRÍPLICE (SELEÇÃO DE 14 VARIÁVEIS CHAVE)")
print("==============================================================================")
print(df_filtro.head(14).to_string())
print(f"\n✅ Total de Variáveis Selecionadas para o Modelo: {len(selected_features)}")

# 3. Treinar Classificador RandomForest (n_estimators=100, max_depth=6)
print("\n⚙️ Treinando RandomForest (n=100, max_depth=6) na Amostra de Treino OOT...")
X_selected_train = df_train[selected_features].fillna(0)
model_scorer = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model_scorer.fit(X_selected_train, y_train)

# 4. Pontuação e Cálculo dos Decis de Risco na Base Completa
X_selected_full = df_bureau[selected_features].fillna(0)
probs_full = model_scorer.predict_proba(X_selected_full)[:, 1]
df_bureau['CALC_PROB_SINISTRO'] = probs_full

try:
    df_bureau['CALC_DECIL_SCORE'] = pd.qcut(df_bureau['CALC_PROB_SINISTRO'], q=10, labels=list(range(1, 11))[::-1])
    df_bureau['CALC_DECIL_SCORE'] = df_bureau['CALC_DECIL_SCORE'].astype(int)
except Exception as e:
    df_bureau['CALC_DECIL_SCORE'] = pd.qcut(df_bureau['CALC_PROB_SINISTRO'].rank(method='first'), q=10, labels=list(range(1, 11))[::-1]).astype(int)

# 5. Validação da Ordenação por Decil e Capacidade Preditiva (AUC-ROC)
probs_tr = model_scorer.predict_proba(X_selected_train)[:, 1]
auc_tr = roc_auc_score(y_train, probs_tr)

print("\n==============================================================================")
print("📊 ORDENAÇÃO DE RISCO POR DECIL NA BASE COMPLETA (CALC_DECIL_SCORE 1 a 10)")
print("==============================================================================")
decil_summary = df_bureau.groupby('CALC_DECIL_SCORE')['ALVO_CASCO'].agg(['count', 'sum', 'mean']).reset_index()
decil_summary.columns = ['CALC_DECIL_SCORE', 'Total_Empresas', 'Qtd_Sinistros', 'Taxa_Sinistro']
decil_summary['Taxa_Sinistro_%'] = (decil_summary['Taxa_Sinistro'] * 100).round(2)
print(decil_summary[['CALC_DECIL_SCORE', 'Total_Empresas', 'Qtd_Sinistros', 'Taxa_Sinistro_%']].to_string(index=False))
print(f"\n📈 Capacidade Preditiva do Modelo de Treino (AUC-ROC): {auc_tr:.4f}")

# 6. GERAÇÃO DOS 3 GRÁFICOS VISUAIS DA ARQUITETURA CRISP-DM
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Gráfico 1: Ordenação da Taxa de Sinistro por Decil de Risco
sns.barplot(data=decil_summary, x='CALC_DECIL_SCORE', y='Taxa_Sinistro_%', ax=axes[0], palette='Reds_r')
axes[0].set_title('Taxa de Sinistro por Decil de Risco (1=Maior, 10=Menor)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Decil de Risco Score')
axes[0].set_ylabel('Taxa de Sinistro (%)')
for p in axes[0].patches:
    height = p.get_height()
    if height > 0:
        axes[0].annotate(f"{height:.1f}%", (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=9)

# Gráfico 2: Curva de Lift e Ganho Acumulado por Decil
decil_summary['Sinistros_Acum_%'] = (decil_summary['Qtd_Sinistros'].cumsum() / decil_summary['Qtd_Sinistros'].sum()) * 100
axes[1].plot(decil_summary['CALC_DECIL_SCORE'], decil_summary['Sinistros_Acum_%'], marker='o', color='navy', linewidth=2.5, label='Ganho Acumulado Modelo')
axes[1].plot([1, 10], [10, 100], color='gray', linestyle='--', label='Modelo Aleatório')
axes[1].set_title('Curva de Ganho Acumulado de Sinistros por Decil', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Decil de Risco Score')
axes[1].set_ylabel('% Acumulada de Sinistros Capturados')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Gráfico 3: Ranking de Importância das 14 Variáveis Selecionadas (Feature Importance)
importances = pd.Series(model_scorer.feature_importances_, index=selected_features).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[2], color='#2b5c8f')
axes[2].set_title('Importância de Atributos na RandomForest (Top 14)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Gini Importance')

plt.tight_layout()
plt.show()


⏰ DIAGNÓSTICO DE TRUNCAMENTO TEMPORAL E SPLIT OUT-OF-TIME
• Base Total Bureau: 3,932 apólices
• Base Madura (até Fev/2024): 3,742 apólices (descartados 190 registros imaturos de Mar-Mai/2024)
• Amostra de Treino (Mai/2020 a Jun/2023): 3,162 apólices | Sinistros: 337 (10.66%)
• Amostra Teste OOT (Jul/2023 a Fev/2024): 580 apólices | Sinistros: 40 (6.90%)

🔍 FILTRO TRÍPLICE DE SELEÇÃO DE VARIÁVEIS (102 → 14 VARIÁVEIS)
                                        Missing_Rate_%  MI_Score  KS_Drift
RAZAO_SITUACAO_MERCADO_ESTADUAL_5_ANOS            3.97    0.0101     0.173
QTD_COLABORADORES                                 0.92    0.0056     0.036
QTD_FECHAMENTOS_GEORREFERENCIADO_5_ANOS           1.32    0.0054     0.026
CONCENTRACAO_SOCIETARIA                           5.14    0.0050     0.024
QTD_ABERTURAS_ESTADUAL_5_ANOS                     3.97    0.0047     0.055
RAZAO_SITUACAO_MERCADO_NACIONAL_5_ANOS            0.92    0.0033     0.185
QTD_FECHAMENTOS_NACIONAL_5_ANOS                   0.92 

## 🏛️ TAREFA 4: Definição da Política de Subscrição em XML (Trillia RiskPack)

Geração dinâmica do arquivo `politica_credito_frotas.xml` seguindo o formato oficial do **Decision Runner / Decision Designer** da Trillia (`<DecisionRunnerNotation>`).

In [14]:
# ==============================================================================
# 🏛️ TAREFA 4: CRIAR ARQUIVO XML DA POLÍTICA (politica_credito_frotas.xml)
# ==============================================================================

xml_content = """<?xml version="1.0" encoding="UTF-8"?>
<DecisionRunnerNotation name="POLITICA_SUBSCRICAO_FROTAS_PJ" version="1.0">
    <Metadata>
        <Description>Politica de Subscricao e Credito Frotas PJ - Ecossistema Trillia RiskPack</Description>
        <Author>Analista de Operacoes e Data Science</Author>
        <Target>PROP_ALVO_CASCO</Target>
    </Metadata>

    <!-- INPUTS (Entradas da Proposta / Bureau com prefixo PROP_) -->
    <Inputs>
        <Input name="PROP_CNPJ" type="String" description="CNPJ da Empresa Proponente" />
        <Input name="PROP_CONCENTRACAO_AREA_RISCO" type="String" description="Nivel de Concentracao Geografica de Risco" />
        <Input name="PROP_QTD_PROCESSOS_JUDICIAIS_REQUERIDO" type="Numeric" description="Quantidade de processos judiciais como requerido" />
        <Input name="PROP_RISCO_LIDERANCA" type="Numeric" description="Indicador de Risco Societario/Lideranca (0 ou 1)" />
        <Input name="PROP_PORTE_FATURAMENTO" type="String" description="Porte presumido do faturamento" />
        <Input name="PROP_SECAO_CNAE" type="String" description="Setor de atividade economica" />
        <Input name="PROP_ALVO_CASCO" type="Numeric" description="Ground truth de sinistro real" />
    </Inputs>

    <!-- PARAMETERS (Parametros de execucao com prefixo VI_) -->
    <Parameters>
        <Parameter name="VI_TICKET_MEDIO_PREMIO" type="Numeric" value="10000.0" />
        <Parameter name="VI_LIMITE_PROCESSOS_EXTREMO" type="Numeric" value="100.0" />
        <Parameter name="VI_LIMITE_PROCESSOS_RESGATE" type="Numeric" value="10.0" />
        <Parameter name="VI_AGRAVAMENTO_D1_PCT" type="Numeric" value="15.0" />
        <Parameter name="VI_AGRAVAMENTO_D2_3_PCT" type="Numeric" value="7.5" />
    </Parameters>

    <!-- DROBJECTS: FLX (Fluxos), RGR (Regras), CALC (Calculos/Outputs) -->
    <DRObjects>
        <Flows>
            <Flow id="FLX_PRINCIPAL" name="Fluxo Principal de Subscricao Frotas">
                <SubFlow ref="FLX_KNOCKOUTS" priority="1" />
                <SubFlow ref="FLX_ROTEAMENTO_SCORE" priority="2" />
            </Flow>
        </Flows>

        <Rules>
            <!-- REGRAS DE KNOCKOUT (RECUSA SUMÁRIA) -->
            <Rule id="RGR_K01_RISCO_LIDERANCA_AREA_CRITICA" decision="RECUSADO" priority="1">
                <Conditions combine="AND">
                    <Condition field="PROP_RISCO_LIDERANCA" operator="EQ" value="1.0" />
                    <Condition field="PROP_CONCENTRACAO_AREA_RISCO" operator="EQ" value="ALTISSIMA" />
                </Conditions>
                <Outcome>
                    <ReasonCode>K01_LIDERANCA_AREA_CRITICA</ReasonCode>
                    <Description>Recusa Sumaria: Risco de lideranca detectado em area de concentracao altissima de sinistro.</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="0.0" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="FALSE" />
                </Outcome>
            </Rule>

            <Rule id="RGR_K02_EXCESSO_PROCESSOS_JUDICIAIS" decision="RECUSADO" priority="2">
                <Conditions combine="AND">
                    <Condition field="PROP_QTD_PROCESSOS_JUDICIAIS_REQUERIDO" operator="GT" value="100.0" />
                </Conditions>
                <Outcome>
                    <ReasonCode>K02_JUDICIAL_EXTREMO</ReasonCode>
                    <Description>Recusa Sumaria: Elevado passivo judicial (> 100 processos como requerido).</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="0.0" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="FALSE" />
                </Outcome>
            </Rule>

            <!-- REGRAS DE ROTEAMENTO HÍBRIDO 3-WAY POR DECIL DE RISCO -->
            <Rule id="RGR_R01_DECIL1_RESGATE_COM_RASTREADOR" decision="ACEITO_COM_CONDICAO" priority="10">
                <Conditions combine="AND">
                    <Condition field="CALC_DECIL_SCORE" operator="EQ" value="1" />
                    <Condition field="PROP_QTD_PROCESSOS_JUDICIAIS_REQUERIDO" operator="LTE" value="10.0" />
                    <Condition field="PROP_CONCENTRACAO_AREA_RISCO" operator="IN" value="BAIXISSIMA,BAIXA,MEDIA" />
                </Conditions>
                <Outcome>
                    <ReasonCode>RECUPERACAO_COMERCIAL_D1</ReasonCode>
                    <Description>Aceite Condicional Decil 1: Perfil de score alto risco, porem mitigado por baixo passivo judicial e geografia favoravel.</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="15.0" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="TRUE" />
                </Outcome>
            </Rule>

            <Rule id="RGR_R02_DECIL1_RECUSA_PADRAO" decision="RECUSADO" priority="11">
                <Conditions combine="AND">
                    <Condition field="CALC_DECIL_SCORE" operator="EQ" value="1" />
                </Conditions>
                <Outcome>
                    <ReasonCode>DECIL1_ALTO_RISCO_SEM_MITIGACAO</ReasonCode>
                    <Description>Recusa por Decil 1 de Risco sem criterios para resgate comercial.</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="0.0" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="FALSE" />
                </Outcome>
            </Rule>

            <Rule id="RGR_R03_DECIL2_3_CONDICIONAL" decision="ACEITO_COM_CONDICAO" priority="20">
                <Conditions combine="AND">
                    <Condition field="CALC_DECIL_SCORE" operator="IN" value="2,3" />
                </Conditions>
                <Outcome>
                    <ReasonCode>ACEITE_CONDICIONAL_DECIL_INTERMEDIARIO</ReasonCode>
                    <Description>Aceite Condicional Decil Intermediario (+7.5% premio).</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="7.5" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="FALSE" />
                </Outcome>
            </Rule>

            <Rule id="RGR_R04_DECIL4_10_APROVADO" decision="APROVADO" priority="30">
                <Conditions combine="AND">
                    <Condition field="CALC_DECIL_SCORE" operator="GTE" value="4" />
                </Conditions>
                <Outcome>
                    <ReasonCode>APROVADO_AUTOMATICO_BAIXO_RISCO</ReasonCode>
                    <Description>Aprovacao Automatica: Risco dentro do apetite da seguradora.</Description>
                    <Action type="CALC_AGRAVAMENTO_PREMIO_PCT" value="0.0" />
                    <Action type="CALC_EXIGE_RASTREADOR_GPS" value="FALSE" />
                </Outcome>
            </Rule>
        </Rules>

        <Calculations>
            <Calculation name="CALC_PROB_SINISTRO" type="Numeric" description="Probabilidade de sinistro gerada pelo modelo ML" />
            <Calculation name="CALC_DECIL_SCORE" type="Integer" description="Decil de risco de 1 a 10" />
            <Calculation name="CALC_DECISAO_FINAL" type="String" description="Decisao final da proposta" />
            <Calculation name="CALC_AGRAVAMENTO_PREMIO_PCT" type="Numeric" description="Porcentagem de sobretaxa aplicada" />
            <Calculation name="CALC_EXIGE_RASTREADOR_GPS" type="Boolean" description="Indicador se rastreador GPS e obrigatorio" />
            <Calculation name="CALC_PREMIO_FINAL_FATURADO" type="Numeric" description="Valor final do premio faturado em BRL" />
            <Calculation name="CALC_RECEITA_ADICIONAL" type="Numeric" description="Receita extra decorrente do agravamento em BRL" />
            <Calculation name="CALC_PERDA_EVITADA" type="Numeric" description="Sinistro evitado atraves da recusa sumaria em BRL" />
        </Calculations>
    </DRObjects>
</DecisionRunnerNotation>"""

# Gravar o arquivo XML no ambiente local
xml_filepath = "politica_credito_frotas.xml"
with open(xml_filepath, "w", encoding="utf-8") as f:
    f.write(xml_content)

print(f"✅ Arquivo da Política XML '{xml_filepath}' gravado com sucesso no ambiente!")


✅ Arquivo da Política XML 'politica_credito_frotas.xml' gravado com sucesso no ambiente!


## ⚡ TAREFA 5: Engine do Motor de Decisão Trillia RiskPack

Construção do executor do motor (`TrilliaXMLPolicyEngine`), responsável por carregar o XML da política e avaliar cada proposta em tempo real.

In [15]:
# ==============================================================================
# ⚡ TAREFA 5: MOTOR DE EXECUÇÃO DA POLÍTICA XML (TrilliaXMLPolicyEngine)
# ==============================================================================

class TrilliaXMLPolicyEngine:
    """Motor de Decisão que lê e avalia políticas em formato XML (DecisionRunnerNotation)."""

    def __init__(self, xml_path: str):
        self.tree = ET.parse(xml_path)
        self.root = self.tree.getroot()
        self.policy_name = self.root.attrib.get('name', 'POLITICA_TRILLIA')

        # Carregar parâmetros da política (VI_)
        self.parameters = {}
        params_elem = self.root.find('Parameters')
        if params_elem is not None:
            for p in params_elem.findall('Parameter'):
                p_name = p.attrib['name']
                p_val = float(p.attrib['value']) if p.attrib.get('type') == 'Numeric' else p.attrib['value']
                self.parameters[p_name] = p_val

    def _avaliar_condicao(self, cond, dados_cnpj: dict) -> bool:
        field = cond.attrib['field']
        operator = cond.attrib['operator']
        target_val = cond.attrib['value']

        # Resolução flexível de campo (aceita 'PROP_VARIAVEL' ou 'VARIAVEL' do CSV)
        val_cnpj = dados_cnpj.get(field, None)
        if val_cnpj is None and field.startswith("PROP_"):
            raw_field = field.replace("PROP_", "", 1)
            val_cnpj = dados_cnpj.get(raw_field, None)

        if pd.isna(val_cnpj) or val_cnpj is None:
            return False

        if operator == 'EQ':
            return str(val_cnpj) == str(target_val) or (float(val_cnpj) == float(target_val) if isinstance(val_cnpj, (int, float)) else False)
        elif operator == 'GT':
            return float(val_cnpj) > float(target_val)
        elif operator == 'GTE':
            return float(val_cnpj) >= float(target_val)
        elif operator == 'LTE':
            return float(val_cnpj) <= float(target_val)
        elif operator == 'IN':
            options = [opt.strip() for opt in target_val.split(',')]
            return str(val_cnpj) in options or (str(int(val_cnpj)) in options if isinstance(val_cnpj, (int, float)) else False)
        return False

    def avaliar_cnpj(self, dados_cnpj: dict) -> dict:
        dr_objects = self.root.find('DRObjects')
        rules_elem = dr_objects.find('Rules') if dr_objects is not None else None

        if rules_elem is not None:
            rules = sorted(rules_elem.findall('Rule'), key=lambda r: int(r.attrib.get('priority', 999)))
            for rule in rules:
                conditions_elem = rule.find('Conditions')
                combine = conditions_elem.attrib.get('combine', 'AND')
                cond_results = [self._avaliar_condicao(c, dados_cnpj) for c in conditions_elem.findall('Condition')]
                rule_fired = all(cond_results) if combine == 'AND' else any(cond_results)

                if rule_fired:
                    outcome = rule.find('Outcome')
                    actions = {act.attrib['type']: act.attrib['value'] for act in outcome.findall('Action')}
                    decision = rule.attrib['decision']
                    agravamento = float(actions.get('CALC_AGRAVAMENTO_PREMIO_PCT', 0.0))

                    # Premissas Financeiras de Mercado (Benchmarking)
                    premio_base = self.parameters.get('VI_TICKET_MEDIO_PREMIO', 10000.0)
                    custo_sinistro = self.parameters.get('VI_CUSTO_MEDIO_SINISTRO', 45000.0)
                    alvo_real = dados_cnpj.get('PROP_ALVO_CASCO', dados_cnpj.get('ALVO_CASCO'))

                    if decision == 'RECUSADO':
                        premio_final = 0.0
                        receita_extra = 0.0
                        perda_evitada = custo_sinistro if alvo_real == 1 else 0.0
                    elif decision == 'ACEITO_COM_CONDICAO':
                        premio_final = premio_base * (1.0 + (agravamento / 100.0))
                        receita_extra = premio_base * (agravamento / 100.0)
                        perda_evitada = 0.0
                    else:  # APROVADO
                        premio_final = premio_base
                        receita_extra = 0.0
                        perda_evitada = 0.0

                    return {
                        'PROP_CNPJ': dados_cnpj.get('PROP_CNPJ', dados_cnpj.get('CNPJ', 'N/A')),
                        'PROP_ALVO_CASCO': alvo_real,
                        'CALC_PROB_SINISTRO': round(dados_cnpj.get('CALC_PROB_SINISTRO', 0.0), 4),
                        'CALC_DECIL_SCORE': dados_cnpj.get('CALC_DECIL_SCORE'),
                        'CALC_DECISAO_FINAL': decision,
                        'RGR_REGRA_DISPARADA_ID': rule.attrib['id'],
                        'CALC_REASON_CODE': outcome.find('ReasonCode').text,
                        'CALC_JUSTIFICATIVA_NEGOCIO': outcome.find('Description').text,
                        'CALC_AGRAVAMENTO_PREMIO_PCT': agravamento,
                        'CALC_EXIGE_RASTREADOR_GPS': str(actions.get('CALC_EXIGE_RASTREADOR_GPS', 'FALSE')).upper() == 'TRUE',
                        'VI_TICKET_MEDIO_PREMIO': premio_base,
                        'CALC_PREMIO_FINAL_FATURADO': premio_final,
                        'CALC_RECEITA_ADICIONAL': receita_extra,
                        'CALC_PERDA_EVITADA': perda_evitada,
                        'PROP_SECAO_CNAE': dados_cnpj.get('SECAO_CNAE'),
                        'PROP_PORTE_FATURAMENTO': dados_cnpj.get('PORTE_FATURAMENTO_PRESUMIDO'),
                        'PROP_CONCENTRACAO_AREA_RISCO': dados_cnpj.get('CONCENTRACAO_AREA_RISCO'),
                        'PROP_QTD_PROCESSOS_REQUERIDO': dados_cnpj.get('QTD_PROCESSOS_JUDICIAIS_REQUERIDO'),
                        'PROP_RISCO_LIDERANCA': dados_cnpj.get('RISCO_LIDERANCA')
                    }
        return {'PROP_CNPJ': dados_cnpj.get('PROP_CNPJ', 'N/A'), 'CALC_DECISAO_FINAL': 'RECUSADO'}

# Teste inicial do Motor
engine_trillia = TrilliaXMLPolicyEngine("politica_credito_frotas.xml")
print(f"✅ Motor Trillia instanciado com sucesso para a política '{engine_trillia.policy_name}'!")


✅ Motor Trillia instanciado com sucesso para a política 'POLITICA_SUBSCRICAO_FROTAS_PJ'!


## 📊 TAREFA 6: Simulação em Lote e Exportação da Base de Insights (CSV)

Execução da simulação sobre todas as empresas do bureau, exportando o arquivo final `resultado_motor_decisao_frotas.csv` no padrão oficial Trillia e gerando os KPIs de impacto financeiro do negócio.

In [16]:
# ==============================================================================
# 📊 TAREFA 6: SIMULAÇÃO EM LOTE E EXPORTAÇÃO DO CSV DE INSIGHTS
# ==============================================================================

# 1. Preparar campos de identificação PROP_
df_bureau['PROP_CNPJ'] = [f"CNPJ_{i+1:06d}" for i in range(len(df_bureau))]
df_bureau['PROP_ALVO_CASCO'] = df_bureau['ALVO_CASCO']

# 2. Executar Motor de Decisão para cada empresa da base
print("⚡ Executando Motor de Decisão Trillia para todas as apólices do Bureau...")
resultados_lote = []
for idx, row in df_bureau.iterrows():
    dados = row.to_dict()
    res = engine_trillia.avaliar_cnpj(dados)
    resultados_lote.append(res)

# 3. Criar DataFrame dos Resultados
df_resultados = pd.DataFrame(resultados_lote)

# 4. Exportar arquivo CSV no padrão Trillia
out_csv = "resultado_motor_decisao_frotas.csv"
df_resultados.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f"\n🎉 [SUCESSO] Base de Resultados salva em CSV: '{out_csv}'")
print(f"• Total de Linhas: {len(df_resultados):,} empresas")
print(f"• Total de Colunas Trillia: {len(df_resultados.columns)} variáveis")

# 5. Apresentação Executiva dos Resultados do Motor (Volumetria Operacional 100% Real do Dataset)
print("\n" + "="*80)
print("📊 RESUMO DE VOLUMETRIA OPERACIONAL DO MOTOR (100% REAL DO DATASET)")
print("="*80)

dist_decisao = df_resultados['CALC_DECISAO_FINAL'].value_counts()
dist_pct = df_resultados['CALC_DECISAO_FINAL'].value_counts(normalize=True) * 100

for dec, count in dist_decisao.items():
    pct = dist_pct[dec]
    print(f"• {dec:<22}: {count:>5} apólices ({pct:5.2f}%)")

gps_count = df_resultados['CALC_EXIGE_RASTREADOR_GPS'].sum()
print(f"\n• Total de frotas de alto risco resgatadas com GPS obrigatório: {gps_count} apólices")

# 6. Projeção Financeira Simulada (Baseada em Premissas Financeiras de Mercado)
print("\n" + "="*80)
print("💰 PROJEÇÃO DE IMPACTO FINANCEIRO (SIMULAÇÃO BASEADA EM PREMISSAS DE BENCHMARK)")
print("ℹ️ Nota de Transparência: A base CSV possui apenas o alvo binário ALVO_CASCO.")
print("ℹ️ Os valores abaixo aplicam as premissas: VI_TICKET_MEDIO_PREMIO = R$ 10.000 e VI_CUSTO_MEDIO_SINISTRO = R$ 45.000.")
print("="*80)

fat_total = df_resultados['CALC_PREMIO_FINAL_FATURADO'].sum()
rec_extra = df_resultados['CALC_RECEITA_ADICIONAL'].sum()
perda_evitada = df_resultados['CALC_PERDA_EVITADA'].sum()

print(f"1. Faturamento Total Emitido (Prêmios) : R$ {fat_total:,.2f}")
print(f"2. Receita Extra por Agravamento (D1-3) : R$ {rec_extra:,.2f}")
print(f"3. Sinistros Evitados (Recusal Sumária) : R$ {perda_evitada:,.2f}")

# 7. Exibir amostra das primeiras 5 linhas do CSV exportado
df_resultados.head(5)


⚡ Executando Motor de Decisão Trillia para todas as apólices do Bureau...

🎉 [SUCESSO] Base de Resultados salva em CSV: 'resultado_motor_decisao_frotas.csv'
• Total de Linhas: 3,932 empresas
• Total de Colunas Trillia: 19 variáveis

📊 RESUMO DE VOLUMETRIA OPERACIONAL DO MOTOR (100% REAL DO DATASET)
• APROVADO              :  2557 apólices (65.03%)
• ACEITO_COM_CONDICAO   :   904 apólices (22.99%)
• RECUSADO              :   471 apólices (11.98%)

• Total de frotas de alto risco resgatadas com GPS obrigatório: 163 apólices

💰 PROJEÇÃO DE IMPACTO FINANCEIRO (SIMULAÇÃO BASEADA EM PREMISSAS DE BENCHMARK)
ℹ️ Nota de Transparência: A base CSV possui apenas o alvo binário ALVO_CASCO.
ℹ️ Os valores abaixo aplicam as premissas: VI_TICKET_MEDIO_PREMIO = R$ 10.000 e VI_CUSTO_MEDIO_SINISTRO = R$ 45.000.
1. Faturamento Total Emitido (Prêmios) : R$ 35,410,250.00
2. Receita Extra por Agravamento (D1-3) : R$ 800,250.00
3. Sinistros Evitados (Recusal Sumária) : R$ 9,990,000.00


,PROP_CNPJ,PROP_ALVO_CASCO,CALC_PROB_SINISTRO,CALC_DECIL_SCORE,CALC_DECISAO_FINAL,RGR_REGRA_DISPARADA_ID,CALC_REASON_CODE,CALC_JUSTIFICATIVA_NEGOCIO,CALC_AGRAVAMENTO_PREMIO_PCT,CALC_EXIGE_RASTREADOR_GPS,VI_TICKET_MEDIO_PREMIO,CALC_PREMIO_FINAL_FATURADO,CALC_RECEITA_ADICIONAL,CALC_PERDA_EVITADA,PROP_SECAO_CNAE,PROP_PORTE_FATURAMENTO,PROP_CONCENTRACAO_AREA_RISCO,PROP_QTD_PROCESSOS_REQUERIDO,PROP_RISCO_LIDERANCA
0,CNPJ_000001,1.0,0.9379,1,ACEITO_COM_CONDICAO,RGR_R01_DECIL1_RESGATE_COM_RASTREADOR,RECUPERACAO_COMERCIAL_D1,Aceite Condicional Decil 1: Perfil de score al...,15.0,True,10000.0,11500.0,1500.0,0.0,G,F,BAIXISSIMA,0.0,0.0
1,CNPJ_000002,1.0,0.7799,1,RECUSADO,RGR_R02_DECIL1_RECUSA_PADRAO,DECIL1_ALTO_RISCO_SEM_MITIGACAO,Recusa por Decil 1 de Risco sem criterios para...,0.0,False,10000.0,0.0,0.0,45000.0,G,NaN,ALTA,0.0,0.0
2,CNPJ_000003,1.0,0.8669,1,ACEITO_COM_CONDICAO,RGR_R01_DECIL1_RESGATE_COM_RASTREADOR,RECUPERACAO_COMERCIAL_D1,Aceite Condicional Decil 1: Perfil de score al...,15.0,True,10000.0,11500.0,1500.0,0.0,G,C,BAIXISSIMA,3.0,1.0
3,CNPJ_000004,1.0,0.9250,1,RECUSADO,RGR_R02_DECIL1_RECUSA_PADRAO,DECIL1_ALTO_RISCO_SEM_MITIGACAO,Recusa por Decil 1 de Risco sem criterios para...,0.0,False,10000.0,0.0,0.0,45000.0,F,D,ALTISSIMA,0.0,0.0
4,CNPJ_000005,1.0,0.7482,1,RECUSADO,RGR_R02_DECIL1_RECUSA_PADRAO,DECIL1_ALTO_RISCO_SEM_MITIGACAO,Recusa por Decil 1 de Risco sem criterios para...,0.0,False,10000.0,0.0,0.0,45000.0,G,NaN,ALTISSIMA,0.0,0.0
